# 05 – Data Quality tầng Gold (Task D04)

**CommerceFlow 360** · Mục báo cáo: 3.7

- Chạy 3 test Gold: `test_gold_fact_grain` (grain 4 fact), `test_gold_dimension_relationships` (khóa fact → dimension),
  `test_sales_reconciliation` (đối soát số đơn, doanh thu, phí vận chuyển, thanh toán Bronze → Silver → Gold).
- Xuất bảng đối soát cho báo cáo: `reports/tables/Bang_3.7.1_DoiSoatGold.csv`.

**Đầu vào:** tầng Gold do `04_Gold_Data_Warehouse.ipynb` tạo.

## 1. Chuẩn bị môi trường

In [1]:
# Trên Colab: clone repository và cài thư viện (bỏ qua khi chạy trong repository trên máy cá nhân)
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not Path('/content/CommerceFlow360').exists():
        !git clone -q https://github.com/huyenpiano1711/CommerceFlow360.git /content/CommerceFlow360
    %cd /content/CommerceFlow360
    %pip install -q -r requirements.txt

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'dbt_project' / 'dbt_project.yml').is_file())
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.project import DUCKDB_PATH, TABLE_DIR, load_test_results, query, run_dbt

assert DUCKDB_PATH.is_file(), 'Chưa có commerceflow.duckdb - chạy notebook 02, 03, 04 trước.'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DUCKDB_PATH :', DUCKDB_PATH)

PROJECT_ROOT: D:\CommerceFlow360
DUCKDB_PATH : D:\CommerceFlow360\commerceflow.duckdb


## 2. Chạy test Gold

In [2]:
GOLD_TESTS = ['test_gold_fact_grain', 'test_gold_dimension_relationships', 'test_sales_reconciliation']
proc = run_dbt('test', '--select', *GOLD_TESTS, check=False)

test_summary = load_test_results()[['test_name', 'status', 'failing_rows']]
display(test_summary)
assert set(GOLD_TESTS) <= set(test_summary['test_name']), 'Thiếu test Gold trong kết quả chạy.'
assert proc.returncode == 0 and test_summary['status'].eq('pass').all(), 'Có test Gold không PASS; xem log dbt ở trên.'

$ dbt test --select test_gold_fact_grain test_gold_dimension_relationships test_sales_reconciliation


16:54:09  Running with dbt=1.12.6
16:54:10  Registered adapter: duckdb=1.11.1
16:54:12  Found 22 models, 89 data tests, 10 sources, 522 macros
16:54:12  
16:54:12  Concurrency: 4 threads (target='dev')
16:54:12  
16:54:13  1 of 3 START test test_gold_dimension_relationships ............................ [RUN]
16:54:13  2 of 3 START test test_gold_fact_grain ......................................... [RUN]
16:54:13  3 of 3 START test test_sales_reconciliation .................................... [RUN]
16:54:13  1 of 3 PASS test_gold_dimension_relationships .................................. [PASS in 0.27s]
16:54:13  2 of 3 PASS test_gold_fact_grain ............................................... [PASS in 0.39s]
16:54:13  3 of 3 PASS test_sales_reconciliation .......................................... [PASS in 0.44s]
16:54:13  
16:54:13  Finished running 3 data tests in 0 hours 0 minutes and 0.82 seconds (0.82s).
16:54:13  
16:54:13  Completed successfully
16:54:13  
16:54:13  Done. PASS=3

,test_name,status,failing_rows
0,test_gold_dimension_relationships,pass,0
1,test_gold_fact_grain,pass,0
2,test_sales_reconciliation,pass,0


## 3. Bảng đối soát cho báo cáo

Cùng phép đối soát với `dbt_project/tests/test_sales_reconciliation.sql`, nhưng trả về toàn bộ số liệu
(test chỉ trả về dòng vi phạm) để đưa vào báo cáo. Sửa một bên thì sửa cả bên kia.

In [3]:
reconciliation = query("""
with reconciliations as (

    select
        'order_count' as metric,
        cast(
            (
                select count(distinct lower(trim(cast(order_id as varchar))))
                from bronze_orders
            ) as double
        ) as source_value,
        cast((select count(*) from silver_orders) as double) as silver_value,
        cast((select sum(order_count) from fact_orders) as double) as gold_value

    union all

    select
        'item_revenue',
        cast((select sum(try_cast(price as decimal(18, 2))) from bronze_order_items) as double),
        cast((select sum(price) from silver_order_items) as double),
        cast((select sum(item_price) from fact_order_items) as double)

    union all

    select
        'freight_value',
        cast((select sum(try_cast(freight_value as decimal(18, 2))) from bronze_order_items) as double),
        cast((select sum(freight_value) from silver_order_items) as double),
        cast((select sum(freight_value) from fact_order_items) as double)

    union all

    select
        'payment_value',
        cast((select sum(try_cast(payment_value as decimal(18, 2))) from bronze_order_payments) as double),
        cast((select sum(payment_value) from silver_order_payments) as double),
        cast((select sum(payment_value) from fact_payments) as double)
)
select
    metric,
    source_value,
    silver_value,
    gold_value,
    abs(source_value - silver_value) as source_to_silver_difference,
    abs(silver_value - gold_value) as silver_to_gold_difference
from reconciliations
order by metric
""")
display(reconciliation)

assert not reconciliation[['source_value', 'silver_value', 'gold_value']].isna().any().any(), 'Có số liệu đối soát bị NULL.'
assert reconciliation[['source_to_silver_difference', 'silver_to_gold_difference']].lt(0.01).all().all(), \
    'Có chênh lệch vượt ngưỡng 0.01.'

TABLE_DIR.mkdir(parents=True, exist_ok=True)
reconciliation.to_csv(TABLE_DIR / 'Bang_3.7.1_DoiSoatGold.csv', index=False, encoding='utf-8-sig')
print('Đã xuất: reports/tables/Bang_3.7.1_DoiSoatGold.csv')

,metric,source_value,silver_value,gold_value,source_to_silver_difference,silver_to_gold_difference
0,freight_value,2251909.54,2251909.54,2251909.54,0.0,0.0
1,item_revenue,13591643.70,13591643.70,13591643.70,0.0,0.0
2,order_count,99441.00,99441.00,99441.00,0.0,0.0
3,payment_value,16008872.12,16008872.12,16008872.12,0.0,0.0


Đã xuất: reports/tables/Bang_3.7.1_DoiSoatGold.csv
